# Fixed-Parameter Sensitivity / Identifiability Analysis

Compute-only notebook: runs `compute_sensitivity` (CasADi analytic derivatives) at
the canonical Bar-Even / Table IV literature parameters and saves all artifacts to
`results/fixed_estimation/`.  NO figures are produced here.

## 1. Imports and setup

In [1]:
import sys
import json
from pathlib import Path

import numpy as np
import pandas as pd

sys.path.insert(0, 'src')

from kinetics_noor import EcoliCarbonKinetics
from sample_parameters import load_theta_init, build_theta_sources_df
from sentitivity import compute_sensitivity

RESULTS_DIR = Path('results/fixed_estimation')
RESULTS_DIR.mkdir(parents=True, exist_ok=True)

print('RESULTS_DIR:', RESULTS_DIR.resolve())


RESULTS_DIR: /Users/gabbi/Desktop/Repos/biosistemas/HybridKinetics-IIQ3733/results/fixed_estimation


## 2. Data import and build model

In [2]:
# Canonical analysis footing -- shared with first_estimation via
# src/utils.py: data-derived metabolite bounds + sorted conditions
# + measurement Q. Guarantees the sensitivities are computed on the same ground as
# first_estimation; only theta differs.
from utils import build_analysis_model, save_per_condition_sensitivity

# Same metabolite-bounds knob as first_estimation (log-space mean +/- N_STD*std).
N_STD = 3

model, valid_conds, input_enzyme, input_cell_needs, measurement_error = build_analysis_model('Data', n_std=N_STD)

print('Model built (data-derived bounds).')
print('  balanced_keys (%d):' % len(model.balanced_keys), model.balanced_keys)
print('  flux_keys     (%d):' % len(model.flux_keys),     model.flux_keys)
print('  conditions (%d, sorted):' % len(valid_conds), valid_conds)


Model built (data-derived bounds).
  balanced_keys (9): ['C_g6p', 'C_f6p', 'C_fbp', 'C_dhap', 'C_g3p', 'C_pgp', 'C_3pg', 'C_2pg', 'C_pep']
  flux_keys     (9): ['v_pts', 'v_pgi', 'v_pfkB', 'v_fbaA', 'v_tpiA', 'v_gapA', 'v_pgk', 'v_gpmA', 'v_eno']
  conditions (22, sorted): ['KO02', 'KO03', 'KO04', 'KO05', 'KO07', 'KO08', 'KO10', 'KO11', 'KO12', 'KO13', 'KO14', 'KO15', 'KO16', 'KO17', 'KO18', 'KO19', 'KO20', 'KO21', 'KO22', 'KO23', 'KO24', 'RF03']


## 3. Load canonical literature theta (seed 0)

In [3]:
theta_lit = load_theta_init('Data/theta_init_sampled.csv')

df_theta = build_theta_sources_df(theta_lit)
print('Loaded %d parameters (seed=0 canonical):' % len(theta_lit))
print(df_theta.to_string())


Loaded 37 parameters (seed=0 canonical):
                  value             source
param                                     
v_max_1       25.739000         literature
Ka1_1          1.000000         literature
Ka2_1          0.010000         literature
Ka3_1          1.000000         literature
K_g6p_1        0.500000         literature
Ks_g6p_pgi     1.018000         literature
Kp_f6p_pgi     0.078000         literature
kcat_f_2      21.820183       sampled_kcat
Ks_f6p_3       0.013000         literature
Ks_atp_3       0.020000         literature
Kp_fbp_3       0.140000         literature
Kp_adp_3       0.162960  sampled_km_linked
kcat_f_3       7.367767       sampled_kcat
Ks_fbp_4       0.240000         literature
Kp_g3p_4       1.696482  sampled_km_direct
Kp_dhap_4      0.829891  sampled_km_direct
kcat_f_4       0.950000         literature
kcat_f_5    1201.200000         literature
Ks_dhap_5      1.030000         literature
Kp_g3p_5       0.197020  sampled_km_direct
kcat_f_6     

## 4. Measurement error (Q)

In [4]:
measured_keys = [k for k in measurement_error if np.isfinite(measurement_error[k])]
print('Measured outputs: %d / %d' % (
    len(measured_keys), len(model.balanced_keys) + len(model.flux_keys)))
print(measured_keys)


Measured outputs: 14 / 18
['C_g6p', 'C_f6p', 'C_fbp', 'C_3pg', 'C_pep', 'v_pts', 'v_pgi', 'v_pfkB', 'v_fbaA', 'v_tpiA', 'v_gapA', 'v_pgk', 'v_gpmA', 'v_eno']


## 5. Inputs and conditions

In [5]:
# compute_sensitivity expects a list of {'condition': name} dicts
conditions = [{'condition': c} for c in valid_conds]
print('Conditions for compute_sensitivity (%d):' % len(conditions), valid_conds)


Conditions for compute_sensitivity (22): ['KO02', 'KO03', 'KO04', 'KO05', 'KO07', 'KO08', 'KO10', 'KO11', 'KO12', 'KO13', 'KO14', 'KO15', 'KO16', 'KO17', 'KO18', 'KO19', 'KO20', 'KO21', 'KO22', 'KO23', 'KO24', 'RF03']


## 6. Compute sensitivity (CasADi analytic)

In [6]:
G_total, corr, diag = compute_sensitivity(
    model             = model,
    params_estimate   = theta_lit,
    input_enzyme      = input_enzyme,
    input_cell_needs  = input_cell_needs,
    conditions        = conditions,
    measurement_error = measurement_error,
)

print('G_total shape:', G_total.shape)
print('corr shape:',    corr.shape)
print('measured outputs (%d):' % len(diag['measured']), diag['measured'])


(308, 37) total measurements across all conditions, 37 parameters.
  FIM condition number: 8.49e+19  (rank-deficient if >> 1e12)
  Near-zero eigenvalues: 20 -- FIM is numerically rank-deficient.
G_total shape: (308, 37)
corr shape: (37, 37)
measured outputs (14): ['C_g6p', 'C_f6p', 'C_fbp', 'C_3pg', 'C_pep', 'v_pts', 'v_pgi', 'v_pfkB', 'v_fbaA', 'v_tpiA', 'v_gapA', 'v_pgk', 'v_gpmA', 'v_eno']


## 6b. Confidence intervals from the covariance (for cross-procedure comparison)

Derive per-parameter confidence intervals from the Cramer-Rao covariance
`cov = pinv(FIM)` computed above, and save them in the **same schema** as the
`confidence_intervals.csv` produced by second/third/fourth_estimation
(`theta, std_err, ci_low, ci_high, cv_percent`) so the confidence coefficients
(CV%, CI widths) are directly comparable across every estimation procedure.

**Caveat for the comparison:** this is the *a-priori* Cramer-Rao bound evaluated
at the *literature* parameters (no fit), from the relative-sensitivity FIM --
whereas the estimation notebooks report the *a-posteriori* covariance at their
fitted theta. So `fixed_estimation` answers "how well could each parameter be
determined from the data at the literature point" -- the identifiability baseline
the fitted procedures are measured against. Non-estimable directions of the
rank-deficient FIM come back as `NaN` (no finite CI).

In [7]:
# CIs from the covariance, in the SAME schema as the parmest estimation notebooks
# (second/third/fourth_estimation) so the confidence coefficients compare directly.
#
# compute_sensitivity built the FIM in the RELATIVE (log-sensitivity) metric, so
# diag['cov'] = pinv(FIM_rel) and diag['stds'] are RELATIVE stds (= CV fractions).
#   cv_percent = 100 * rel_std          (scale-free -- the coefficient to compare)
#   std_err    = rel_std * |theta|      (absolute std, parameter units)
#   ci_low/high = theta -/+ z * std_err (z = 1.96 for a 95% two-sided interval)
# Non-estimable directions (null space of the rank-deficient a-priori FIM) are NaN.
from scipy.stats import norm

params = list(model.params_keys)
theta_vec = np.array([theta_lit[k] for k in params], dtype=float)
rel_std = np.asarray(diag['stds'], dtype=float)        # relative std, NaN if non-estimable
z = float(norm.ppf(0.975))                             # 1.95996..., 95% two-sided

std_err = rel_std * np.abs(theta_vec)                  # absolute std (parameter units)
ci_df = pd.DataFrame({
    'theta':      theta_vec,
    'std_err':    std_err,
    'ci_low':     theta_vec - z * std_err,
    'ci_high':    theta_vec + z * std_err,
    'cv_percent': 100.0 * rel_std,
}, index=params)
ci_df.to_csv(RESULTS_DIR / 'confidence_intervals.csv')
print('saved confidence_intervals.csv', ci_df.shape,
      '(%d estimable / %d params)' % (int(np.isfinite(rel_std).sum()), len(params)))

# Covariance in ABSOLUTE parameter units (comparable to the other notebooks'
# covariance.csv):  cov_abs[i,j] = cov_rel[i,j] * theta_i * theta_j.
cov_abs = np.outer(theta_vec, theta_vec) * np.asarray(diag['cov'], dtype=float)
cov_df = pd.DataFrame(cov_abs, index=params, columns=params)
cov_df.to_csv(RESULTS_DIR / 'covariance.csv')
print('saved covariance.csv', cov_df.shape, '(absolute parameter units)')

# The relative-sensitivity FIM the CIs come from (provenance / re-analysis).
fim_df = pd.DataFrame(np.asarray(diag['FIM'], dtype=float), index=params, columns=params)
fim_df.to_csv(RESULTS_DIR / 'fim.csv')
print('saved fim.csv', fim_df.shape, '(relative / log-sensitivity FIM)')

print()
print(ci_df.round(4).to_string())


saved confidence_intervals.csv (37, 5) (37 estimable / 37 params)
saved covariance.csv (37, 37) (absolute parameter units)
saved fim.csv (37, 37) (relative / log-sensitivity FIM)

                theta      std_err       ci_low      ci_high  cv_percent
v_max_1       25.7390    1636.1015   -3180.9610    3232.4390   6356.5076
Ka1_1          1.0000      62.6277    -121.7479     123.7479   6262.7650
Ka2_1          0.0100       8.9038     -17.4410      17.4610  89037.5373
Ka3_1          1.0000      70.7907    -137.7472     139.7472   7079.0664
K_g6p_1        0.5000       0.0000       0.4999       0.5001      0.0065
Ks_g6p_pgi     1.0180       1.4554      -1.8346       3.8706    142.9687
Kp_f6p_pgi     0.0780       2.3299      -4.4884       4.6444   2986.9940
kcat_f_2      21.8202      49.0561     -74.3280     117.9683    224.8197
Ks_f6p_3       0.0130       0.0174      -0.0210       0.0470    133.4783
Ks_atp_3       0.0200       0.0267      -0.0323       0.0723    133.4780
Kp_fbp_3       0.

## 7. Save artifacts

In [8]:
# correlation.csv (a-priori, relative-sensitivity FIM at literature theta)
corr_df = pd.DataFrame(corr, index=model.params_keys, columns=model.params_keys)
corr_df.to_csv(RESULTS_DIR / 'correlation.csv')
print('saved correlation.csv', corr_df.shape)

# per-condition absolute sensitivity CSVs + predictions/real on the SHARED footing
pred_df, real_df = save_per_condition_sensitivity(
    model, theta_lit, valid_conds, input_enzyme, input_cell_needs, RESULTS_DIR, data_dir='Data')
pred_df.to_csv(RESULTS_DIR / 'predictions.csv')
real_df.to_csv(RESULTS_DIR / 'real.csv')
print('saved sensitivity/ (%d conds) + predictions.csv %s + real.csv %s'
      % (len(valid_conds), pred_df.shape, real_df.shape))

# correlated_pairs.csv (|r| >= 0.9)
pairs = []
for i in range(len(model.params_keys)):
    for j in range(i + 1, len(model.params_keys)):
        r = corr[i, j]
        if np.isfinite(r) and abs(r) >= 0.9:
            pairs.append({'Parameter 1': model.params_keys[i],
                          'Parameter 2': model.params_keys[j],
                          'Correlation': r})
pairs_df = pd.DataFrame(pairs, columns=['Parameter 1', 'Parameter 2', 'Correlation'])
pairs_df.to_csv(RESULTS_DIR / 'correlated_pairs.csv', index=False)
print('saved correlated_pairs.csv (%d pairs with |r| >= 0.9)' % len(pairs_df))

# theta_lit.csv (value column for plots_fixed_estimation)
df_theta.to_csv(RESULTS_DIR / 'theta_lit.csv', index_label='param')
print('saved theta_lit.csv')

# manifest.json
manifest = {
    'seed':         0,
    'mode':         'literature; correlation analysis (compute_sensitivity, CasADi analytic)',
    'source':       'Data/theta_init_sampled.csv',
    'n_conditions': len(valid_conds),
    'conditions':   list(valid_conds),
    'n_measured':   len(measured_keys),
    'confidence_intervals': 'a-priori Cramer-Rao from relative-sensitivity FIM at literature theta',
}
with open(RESULTS_DIR / 'manifest.json', 'w') as fh:
    json.dump(manifest, fh, indent=2)
print('saved manifest.json')

print()
print('=== results/fixed_estimation/ ===')
for p in sorted(RESULTS_DIR.iterdir()):
    print(' ', p.name)


saved correlation.csv (37, 37)


saved sensitivity/ (22 conds) + predictions.csv (22, 18) + real.csv (22, 18)
saved correlated_pairs.csv (83 pairs with |r| >= 0.9)
saved theta_lit.csv
saved manifest.json

=== results/fixed_estimation/ ===
  .DS_Store
  confidence_intervals.csv
  correlated_pairs.csv
  correlation.csv
  covariance.csv
  figures
  fim.csv
  manifest.json
  predictions.csv
  real.csv
  sensitivity
  theta_lit.csv
